In [ ]:
%%capture
!pip install unsloth wandb peft

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # P100 (sm_60) incompatible with current torch; run on CPU

# ── Config ────────────────────────────────────────────────────────────────────
BASE_MODEL_ID  = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit"
LORA_REPO      = "docvm/sakhi-gemma4-e2b-asha-lora"
# Add maternal_triage_cases.json as a Kaggle dataset input named "sakhi-eval"
CASES_PATH     = "/kaggle/input/sakhi-eval/maternal_triage_cases.json"
MAX_SEQ_LEN    = 512
MAX_NEW_TOKENS = 300
WANDB_PROJECT  = "sakhi-eval"
RESULTS_DIR    = "/kaggle/working"  # intermediate results saved here
DEVICE         = "cpu"

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

secrets = UserSecretsClient()
login(secrets.get_secret("HF_TOKEN"))
wandb.login(key=secrets.get_secret("WANDB_TOKEN"))

In [ ]:
import json
from collections import Counter

with open(CASES_PATH) as f:
    cases = json.load(f)

dist = Counter(c["guideline_triage"] for c in cases)
print(f"Loaded {len(cases)} cases")
print(f"Distribution: {dict(dist)}")

In [ ]:
import re

def prompt_from_case(case):
    v = case["vitals"]
    vitals_parts = [f"BP {v['bp']}"]
    if v.get("temperature"):
        vitals_parts.append(f"temp {v['temperature']}")
    sugar = v.get("blood_sugar", "")
    if sugar and sugar != "Not available":
        vitals_parts.append(f"blood sugar {sugar}")

    text = (
        f"I visited a {case['age']}-year-old woman who is {case['gestational_stage']}.\n"
        f"Vitals: {', '.join(vitals_parts)}.\n"
        f"She reports: {case['symptoms']}\n"
        f"History: {case['history']}\n"
        "What is the risk level and what should I do?"
    )
    return (
        "<start_of_turn>user\n"
        f"{text}\n"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )


def parse_risk(response: str) -> str:
    m = re.search(r"\*\*Risk Level:\*\*\s*([\w][\w\s]*?)(?:\n|\*|$)", response, re.IGNORECASE)
    if not m:
        return "PARSE_FAIL"
    text = m.group(1).strip().lower()
    if "high" in text:
        return "HIGH RISK"
    if "mid" in text or "moderate" in text or "medium" in text:
        return "MODERATE RISK"
    if "low" in text:
        return "LOW RISK"
    if "n/a" in text or text == "na":
        return "N/A"
    return "UNKNOWN"

In [ ]:
import os
import torch
from tqdm import tqdm
from unsloth import FastModel


def run_eval(model, tokenizer, cases, save_path=None):
    FastModel.for_inference(model)
    results = []

    # Resume from partial save if it exists
    if save_path and os.path.exists(save_path):
        with open(save_path) as f:
            results = json.load(f)
        done_ids = {r["case_id"] for r in results}
        remaining = [c for c in cases if c["case_id"] not in done_ids]
        print(f"Resuming: {len(results)}/{len(cases)} already done, {len(remaining)} remaining")
    else:
        remaining = cases

    for case in tqdm(remaining, total=len(cases), initial=len(results)):
        prompt = prompt_from_case(case)
        inputs = tokenizer(text=prompt, return_tensors="pt").to(DEVICE)
        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.2,
                do_sample=True,
            )
        response = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
        results.append({
            "case_id":           case["case_id"],
            "gestational_stage": case["gestational_stage"],
            "symptoms":          case["symptoms"],
            "true_risk":         case["guideline_triage"],
            "predicted_risk":    parse_risk(response),
            "response":          response,
        })
        # Save after every case so a crash loses at most one result
        if save_path:
            with open(save_path, "w") as f:
                json.dump(results, f)

    return results

In [ ]:
import gc

BASE_SAVE = f"{RESULTS_DIR}/base_results.json"

if os.path.exists(BASE_SAVE) and json.load(open(BASE_SAVE)).__len__() == len(cases):
    print("Loading cached base results")
    with open(BASE_SAVE) as f:
        base_results = json.load(f)
else:
    print("── Evaluating base model ──")
    base_model, tokenizer = FastModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
    )
    base_results = run_eval(base_model, tokenizer, cases, save_path=BASE_SAVE)
    del base_model
    gc.collect()
    print("Base eval done, model unloaded")

In [ ]:
FT_SAVE = f"{RESULTS_DIR}/ft_results.json"

if os.path.exists(FT_SAVE) and json.load(open(FT_SAVE)).__len__() == len(cases):
    print("Loading cached fine-tuned results")
    with open(FT_SAVE) as f:
        ft_results = json.load(f)
else:
    print("── Evaluating fine-tuned model ──")
    ft_model, tokenizer = FastModel.from_pretrained(
        model_name=LORA_REPO,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
    )
    ft_results = run_eval(ft_model, tokenizer, cases, save_path=FT_SAVE)
    del ft_model
    gc.collect()
    print("Fine-tuned eval done, model unloaded")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

LABELS = ["HIGH RISK", "MODERATE RISK", "LOW RISK"]


def compute_metrics(results, prefix):
    true = [r["true_risk"] for r in results]
    pred = [r["predicted_risk"] for r in results]

    parse_fails = sum(1 for p in pred if p not in LABELS)
    valid = [(t, p) for t, p in zip(true, pred) if p in LABELS]
    t_v, p_v = zip(*valid) if valid else ([], [])

    acc = accuracy_score(t_v, p_v) if t_v else 0.0
    report = classification_report(t_v, p_v, labels=LABELS, output_dict=True, zero_division=0)

    high_as_low = sum(1 for t, p in zip(true, pred) if t == "HIGH RISK" and p == "LOW RISK")
    n_high = sum(1 for t in true if t == "HIGH RISK")

    print(f"\n{'='*40} {prefix} {'='*40}")
    print(f"  Overall accuracy : {acc:.1%}  ({len(valid)}/{len(results)} parseable)")
    print(f"  Safety failures  : {high_as_low}/{n_high} HIGH predicted as LOW  "
          f"({high_as_low/n_high:.1%})" if n_high else "  Safety failures  : 0")
    print(f"  Parse fails      : {parse_fails}")
    print()
    print(classification_report(t_v, p_v, labels=LABELS, zero_division=0))

    return {
        f"{prefix}/accuracy":            acc,
        f"{prefix}/safety_failure_rate": high_as_low / n_high if n_high else 0,
        f"{prefix}/safety_failures":     high_as_low,
        f"{prefix}/parse_fails":         parse_fails,
        **{f"{prefix}/{lbl.split()[0].lower()}_f1": report[lbl]["f1-score"]
           for lbl in LABELS if lbl in report},
    }


base_metrics = compute_metrics(base_results, "base")
ft_metrics   = compute_metrics(ft_results,   "ft")

In [ ]:
run = wandb.init(project=WANDB_PROJECT, job_type="eval", config={
    "base_model": BASE_MODEL_ID,
    "lora_repo":  LORA_REPO,
    "n_cases":    len(cases),
})

# Summary scalars
wandb.log({**base_metrics, **ft_metrics})

# Per-case predictions table
cols = [
    "case_id", "gestational_stage", "symptoms", "true_risk",
    "base_pred", "ft_pred",
    "base_correct", "ft_correct",
    "base_safety_fail", "ft_safety_fail",
    "base_response", "ft_response",
]
rows = []
for b, f in zip(base_results, ft_results):
    rows.append([
        b["case_id"],
        b["gestational_stage"],
        b["symptoms"][:150],
        b["true_risk"],
        b["predicted_risk"],
        f["predicted_risk"],
        b["predicted_risk"] == b["true_risk"],
        f["predicted_risk"] == f["true_risk"],
        b["true_risk"] == "HIGH RISK" and b["predicted_risk"] == "LOW RISK",
        f["true_risk"] == "HIGH RISK" and f["predicted_risk"] == "LOW RISK",
        b["response"],
        f["response"],
    ])

wandb.log({"predictions": wandb.Table(columns=cols, data=rows)})

# Confusion matrices
for label, results in [("base", base_results), ("ft", ft_results)]:
    parseable = [(r["true_risk"], r["predicted_risk"]) for r in results if r["predicted_risk"] in LABELS]
    if parseable:
        t_labels, p_labels = zip(*parseable)
        wandb.log({
            f"{label}/confusion_matrix": wandb.plot.confusion_matrix(
                y_true=list(t_labels),
                preds=list(p_labels),
                class_names=LABELS,
            )
        })

wandb.finish()
print(f"\nResults → {run.url}")